In [ ]:
#xgboost notebook

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import re

In [13]:
# Define file paths
file_path_train = '/content/drive/My Drive/Colab Notebooks/Data/credit-risk/'

In [20]:
# Load the dataset
df = pd.read_excel(file_path_train + 'output.xlsx')

df.replace(['n.a.', 'n.s.'], np.nan, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
print(df.shape)  # Should give (total_rows, total_columns)


(5506, 62)


In [19]:
# Define features (X) and target variable (y)
X = df.drop(['Company name Latin alphabet','Status Updated'], axis=1)  # Features
y = df['Status Updated']
y = np.where(df['Status Updated'] == 'Active', 1, 0)

print(df.shape)  # Should give (total_rows, total_columns)
# Rename the column before one-hot encoding
X = X.rename(columns={'NACE Rev. 2, core code (4 digits)': 'NACE'})

# One-hot encode categorical features before train_test_split
categorical_cols = ['Quoted', 'Branch', 'Woco', 'NACE',
                    'Consolidation code', 'Standardized country']

# One-hot encode the specified categorical features
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Fix: Replace problematic characters in all column names
for column in X.columns:
    new_column = re.sub(r'[^a-zA-Z0-9_]', '_', column)  # Replace all non-alphanumeric and underscore characters
    X = X.rename(columns={column: new_column})



print(X.shape)  # Should give (total_rows, total_columns)


(5506, 62)
(5506, 169)


In [21]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42) # random_state for reproducibility

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(3854, 169) (3854,)
(1652, 169) (1652,)


In [22]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV

# Define the XGBoost regressor with GPU support
xgb_regressor = xgb.XGBRegressor(
    tree_method='hist',
    device='cuda',
    random_state=42
)

# Hyperparameter grid for XGBoost
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],  # Optional: you can also tune this
    'colsample_bytree': [0.8, 1.0]  # Optional: you can also tune this
}

# GridSearchCV using the pre-defined xgb_regressor
grid_search = GridSearchCV(estimator=xgb_regressor,  # Use the model with GPU support
                           param_grid=param_grid,
                           scoring='neg_mean_squared_error',
                           cv=5)


# Fit grid search
grid_search.fit(X_train, y_train)

# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Output the best parameters
print(f"Best Parameters from GridSearchCV: {best_params}")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [15:47:36] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)


Best Parameters from GridSearchCV: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}


In [23]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import numpy as np

# Initialize and train the XGBoost regressor
model = XGBRegressor(objective='reg:squarederror', random_state=42)
model.fit(X_train, y_train)

# Train the model using the best parameters from GridSearchCV
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model using Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

# Print the results
print(f"Optimized XGBoost MSE: {mse}")
print(f"Optimized XGBoost RMSE: {rmse}")

# Cross-validation for model performance evaluation
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
mse_cv = -cv_scores.mean()  # Negate since sklearn returns negative MSE
rmse_cv = np.sqrt(mse_cv)
# Compute MAE
mae = mean_absolute_error(y_test, y_pred)

print(f"Mean Absolute Error: {mae}")
print(f"Cross-validated MSE: {mse_cv}")
print(f"Cross-validated RMSE: {rmse_cv}")


Optimized XGBoost MSE: 0.14360611140727997
Optimized XGBoost RMSE: 0.37895397003762865
Mean Absolute Error: 0.3048345148563385
Cross-validated MSE: 0.15749566853046418
Cross-validated RMSE: 0.39685723948350016


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# Make predictions on the test set (convert to binary if necessary)
y_pred_binary = [1 if p >= 0.5 else 0 for p in y_pred]  # Adjust threshold if needed

# Calculate classification metrics
accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)
conf_matrix = confusion_matrix(y_test, y_pred_binary)

# Print the results
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Confusion Matrix:\n{conf_matrix}")

Accuracy: 0.8026634382566586
Precision: 0.8054711246200608
Recall: 0.8557588805166846
F1 Score: 0.8298538622129437
Confusion Matrix:
[[531 192]
 [134 795]]
